# 課程 08 - 多代理設計模式


## 設置


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 為什麼選擇多智能體系統？

現實世界的任務如行程規劃涉及多種專業知識 —— 物流、本地知識、預算等。單一智能體嘗試處理所有事情很快會變得難以應付。

多智能體系統透過**專業化**來解決這個問題：每個智能體專注於一個專業領域，產出比通才更高質素的結果。它們亦提升了**擴展性** —— 你可以新增智能體（例如，飛行專家、餐廳評論家）而無需重寫現有的工作流程。這些智能體通過結構化的管道組合起來，將上下文從一個傳遞到下一個。


## 創建專門代理


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立序列式工作流程

`WorkflowBuilder` 讓你可將代理串接成有向圖。我們在這裡建立一個簡單的兩步驟流程：**TravelPlanner** 草擬行程，接著由 **TravelConcierge** 審核並加以提升。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 新增更多代理到工作流程

多代理模式的最大優點之一就是擴展簡單。以下我們新增了一個 **BudgetReviewer** 代理，負責檢查計劃是否符合旅客的預算，標記可能超出預算的項目，並建議省錢的替代方案。現在工作流程依序運行三個代理：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

在本課程中，您學會了如何：

1. **建立專門的代理人** — 每個代理人都有專注的角色（規劃、禮賓、預算審查）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理人串聯成順序工作流程。**
3. **從多代理人流程中串流輸出，追蹤哪個代理人在說話。**
4. **擴展工作流程，透過在鏈中添加新的代理人而不修改現有代理人。**

多代理人設計模式讓每個代理人都保持簡單，同時產生比單一代理人更豐富、更徹底審查的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件經由人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 製作。儘管我們努力確保翻譯的準確性，但自動翻譯可能包含錯誤或不準確之處。原文檔案及其母語版本應被視為最具權威性的來源。對於重要資訊，建議尋求專業人員進行人工翻譯。我們對因使用本翻譯而產生的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
